### Train model untuk Labeling Data

In [1]:
import os
import sys
import pandas as pd
import pymongo
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, when
from pyspark.ml.feature import Tokenizer, HashingTF, IDF
from pyspark.ml.classification import LogisticRegression
from pyspark.ml import Pipeline
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

# Konfigurasi Environment
os.environ['PYSPARK_PYTHON'] = sys.executable
os.environ['PYSPARK_DRIVER_PYTHON'] = sys.executable
os.environ['HADOOP_HOME'] = "C:\\hadoop"
sys.path.append("C:\\hadoop\\bin")

spark = SparkSession.builder.appName("Ewallet_Modeling").getOrCreate()

# Tarik Data Emas dari MongoDB
client = pymongo.MongoClient("mongodb://127.0.0.1:27017/")
db = client["capstone_db"]
df_pd = pd.DataFrame(list(db["ground_truth_1000"].find({}, {'_id':0})))
client.close()

# Masukkan ke Spark DataFrame
df_spark = spark.createDataFrame(df_pd)
df_spark = df_spark.withColumn("label", col("label_sentimen").cast("double")) 

# =========================================================
# FIX ERROR: UBAH LABEL NEGATIF (-1) MENJADI 2
# 0 = Netral, 1 = Positif, 2 = Negatif
# =========================================================
df_spark = df_spark.withColumn("label", when(col("label") == -1.0, 2.0).otherwise(col("label")))

# Bagi Data: 80% untuk Belajar, 20% untuk Uji
train_data, test_data = df_spark.randomSplit([0.8, 0.2], seed=42)
print(f"Data Latih: {train_data.count()} baris | Data Uji: {test_data.count()} baris")

# Bangun Arsitektur NLP
tokenizer = Tokenizer(inputCol="clean_content", outputCol="words")
hashingTF = HashingTF(inputCol=tokenizer.getOutputCol(), outputCol="rawFeatures", numFeatures=5000)
idf = IDF(inputCol=hashingTF.getOutputCol(), outputCol="features")
lr = LogisticRegression(featuresCol="features", labelCol="label", maxIter=20)

pipeline = Pipeline(stages=[tokenizer, hashingTF, idf, lr])

# Eksekusi Model
print("Sedang melatih model Logistic Regression...")
model = pipeline.fit(train_data)

# Uji Akurasi
predictions = model.transform(test_data)
evaluator = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="accuracy")
accuracy = evaluator.evaluate(predictions)

print("="*40)
print(f"🎯 AKURASI MODEL AWAL: {accuracy * 100:.2f}%")
print("="*40)

Data Latih: 805 baris | Data Uji: 194 baris
Sedang melatih model Logistic Regression...
🎯 AKURASI MODEL AWAL: 75.77%


### Auto Labeling

In [3]:
# ====================================================================
# TAHAP HYBRID LABELING: MENGGUNAKAN AI UNTUK MELABELI SISA DATA
# ====================================================================
import pymongo
import pandas as pd
from pyspark.sql.functions import col, when

# 1. Tarik seluruh data bersih dari MongoDB (sekitar 13.000+ data)
print("1. Menarik seluruh data bersih dari MongoDB...")
client = pymongo.MongoClient("mongodb://127.0.0.1:27017/")
db = client["capstone_db"]
df_semua_data = pd.DataFrame(list(db["clean_ewallet_data"].find({}, {'_id':0})))

# 2. Masukkan ke dalam mesin Spark
df_semua_spark = spark.createDataFrame(df_semua_data)

# 3. Lakukan PREDIKSI OTOMATIS menggunakan model Logistic Regression tadi
print("2. AI sedang membaca dan melabeli 12.000+ data. Mohon tunggu...")
hasil_prediksi = model.transform(df_semua_spark)

# 4. Kembalikan label agar sesuai dengan format awal kamu
# (Ubah angka 2.0 kembali menjadi -1 untuk sentimen negatif)
hasil_prediksi = hasil_prediksi.withColumn(
    "label_sentimen", 
    when(col("prediction") == 2.0, -1)
    .otherwise(col("prediction").cast("int"))
)

# 5. Pilih kolom-kolom penting saja untuk disimpan (SUDAH DIPERBAIKI)
data_final_spark = hasil_prediksi.select(
    "source",        # Asal data (Playstore/Quora/Youtube)
    "app_name",      # Nama E-wallet
    "score",         # Rating bintang (menggantikan "rating")
    "at",            # Waktu review
    "content",       # Teks asli
    "clean_content", # Teks bersih
    "label_sentimen" # Hasil tebakan AI
)

# 6. Simpan hasil akhirnya kembali ke MongoDB di koleksi yang baru!
print("3. Mengirim data yang sudah dilabeli penuh ke MongoDB...")
records_final = data_final_spark.toPandas().to_dict('records')

koleksi_final = db["final_labeled_data"]
koleksi_final.delete_many({}) # Bersihkan jika sebelumnya ada
koleksi_final.insert_many(records_final)

client.close()
print("="*50)
print(f"🚀 SUKSES BESAR! {len(records_final)} data kini telah memiliki label!")
print("Silakan cek koleksi 'final_labeled_data' di MongoDB Compass kamu.")
print("="*50)

1. Menarik seluruh data bersih dari MongoDB...
2. AI sedang membaca dan melabeli 12.000+ data. Mohon tunggu...
3. Mengirim data yang sudah dilabeli penuh ke MongoDB...
🚀 SUKSES BESAR! 12375 data kini telah memiliki label!
Silakan cek koleksi 'final_labeled_data' di MongoDB Compass kamu.
